# CV02 — Suy luận quy luật thị giác

Tám ô ngữ cảnh và A đáp án. Mỗi đáp án được ghép thành một chuỗi 9 ô, chấm điểm riêng; dùng CE trên A điểm. Split theo template/quy luật để phát hiện shortcut.

Năm cell code, chạy lần lượt. Fixture không phải benchmark; xem REFERENCE.md để chọn dữ liệu thật và paper gốc.


## Cell 1 — Môi trường và seed

Dùng CPU để kiểm tra trước. Tìm repo root từ thư mục notebook, đặt seed và import PyTorch. Không tải pretrained weights.


In [ ]:
from pathlib import Path
import sys
import torch
from torch import nn
import torch.nn.functional as F

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / 'vision/curriculum').is_dir())
sys.path.insert(0, str(ROOT))
torch.manual_seed(42)
torch.set_num_threads(2)
device = torch.device('cpu')  # Change to cuda after the CPU example works.
print('Device:', device)


## Cell 2 — Tạo batch và model

Tám ô ngữ cảnh và A đáp án. Mỗi đáp án được ghép thành một chuỗi 9 ô, chấm điểm riêng; dùng CE trên A điểm. Split theo template/quy luật để phát hiện shortcut. Dữ liệu trong cell là fixture tổng hợp để bạn đọc được shape; thay bằng Dataset thật trước khi so metric.


In [ ]:
from vision.curriculum.cv02_reasoning import Reasoner, loss_fn
context = torch.rand(2, 8, 3, 32, 32, device=device)
choices = torch.rand(2, 4, 3, 32, 32, device=device)
labels = torch.tensor([0, 1], device=device)
model = Reasoner(mode='relation').to(device)
batch = (context, choices, labels)


## Cell 3 — Forward và baseline

Đọc shape/output trước khi train. Các số in ra chỉ minh họa phép tính và luồng dự đoán.


In [ ]:
model.eval()
with torch.no_grad():
    scores = model(context, choices)
print('Scores B x A:', scores.shape)
print('Candidate indices:', scores.argmax(-1).tolist())


## Cell 4 — Một bước cập nhật

Tính đúng loss của nhiệm vụ, backward và cập nhật tham số. Một bước không phải quy trình huấn luyện hoàn chỉnh. TENT tự thực hiện update bên trong adapt.


In [ ]:
model.train()
optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=3e-4)
optimizer.zero_grad(set_to_none=True)
loss = loss_fn(model, *batch)
loss.backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
optimizer.step()
print('One-step training loss:', loss.detach().item())


## Cell 5 — Chẩn đoán và hướng nâng cấp

MLP trên 9 embeddings → all-pairs relation → Transformer. Phép đổi thứ tự đáp án phải đổi đúng thứ tự score; accuracy trên fixture ngẫu nhiên không có ý nghĩa.


In [ ]:
model.eval()
with torch.no_grad():
    permutation = torch.tensor([2, 0, 3, 1], device=device)
    original = model(context, choices)
    permuted = model(context, choices[:, permutation])
    error = (permuted - original[:, permutation]).abs().max()
print('Candidate permutation error:', float(error))
print('Baseline -> relation -> transformer: change mode; retrain on the same split.')


## Thử nghiệm tiếp theo

MLP trên 9 embeddings → all-pairs relation → Transformer. Phép đổi thứ tự đáp án phải đổi đúng thứ tự score; accuracy trên fixture ngẫu nhiên không có ý nghĩa.

Giữ nguyên split và ngân sách huấn luyện. Đọc [module](../cv02_reasoning.py) và [tài liệu gốc](../REFERENCE.md).
